In [1]:
!pip install -U scikit-learn==1.5.0
!pip install -U imbalanced-learn==0.13.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 52.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 7.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sklearn
import imblearn

print("scikit-learn version:", sklearn.__version__)
print("imblearn version:", imblearn.__version__)

scikit-learn version: 1.5.0
imblearn version: 0.13.0


In [4]:
!pip install polars[numpy,pandas,pyarrow] --no-index --find-links=file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg

Looking in links: file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg
Processing /kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg/polars-0.20.16-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [5]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import glob
import os
import polars as pl
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import pickle
import ctypes
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

E0000 00:00:1745167071.910761      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


In [6]:
# detect TPUs
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

print("Number of accelerators: ", tpu_strategy.num_replicas_in_sync)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local


I0000 00:00:1745167156.086092      10 service.cc:148] XLA service 0x5d077c978e40 initialized for platform TPU (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745167156.086140      10 service.cc:156]   StreamExecutor device (0): TPU, 2a886c8
I0000 00:00:1745167156.086144      10 service.cc:156]   StreamExecutor device (1): TPU, 2a886c8
I0000 00:00:1745167156.086147      10 service.cc:156]   StreamExecutor device (2): TPU, 2a886c8
I0000 00:00:1745167156.086150      10 service.cc:156]   StreamExecutor device (3): TPU, 2a886c8
I0000 00:00:1745167156.086153      10 service.cc:156]   StreamExecutor device (4): TPU, 2a886c8
I0000 00:00:1745167156.086156      10 service.cc:156]   StreamExecutor device (5): TPU, 2a886c8
I0000 00:00:1745167156.086158      10 service.cc:156]   StreamExecutor device (6): TPU, 2a886c8
I0000 00:00:1745167156.086161      10 service.cc:156]   StreamExecutor device (7): TPU, 2a886c8


INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:5, TPU, 0, 0)
I

In [7]:
feature_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/extracted_train_feat_five_sec_v5_2025'
label_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/labels_five_sec_v5_2025'

missing_classes_feature_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_features_2025'
missing_classes_label_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_labels_2025'

with open(feature_file_path, "rb") as file:
    pickled_extracted_features_five_sec = pickle.load(file)
    
with open(label_file_path, "rb") as file:
    labels_five_sec = pickle.load(file)

with open(missing_classes_feature_file_path, "rb") as file:
    pickled_missing_classes_features_five_sec = pickle.load(file)
    
with open(missing_classes_label_file_path, "rb") as file:
    labels_missing_classes_five_sec = pickle.load(file)

In [10]:
print("Previous labels shape:", labels_five_sec.shape)
print("Previous labels dtype:", labels_five_sec.dtype)
print("Previous labels sample:", labels_five_sec[:5])  # Show first 5 elements

print("\nNew labels shape:", labels_missing_classes_five_sec.shape)
print("New labels dtype:", labels_missing_classes_five_sec.dtype)
print("New labels sample:", labels_missing_classes_five_sec[:5])

Previous labels shape: (180142,)
Previous labels dtype: int64
Previous labels sample: [110 110 110 110 110]

New labels shape: (620,)
New labels dtype: int64
New labels sample: [17 17 17 17 17]


In [11]:
x_five_sec = np.vstack([pickled_extracted_features_five_sec, pickled_missing_classes_features_five_sec])
y_five_sec = np.concatenate([labels_five_sec, labels_missing_classes_five_sec], axis=0)

print("Combined features shape:", x_five_sec.shape)
print("Combined labels shape:", y_five_sec.shape)

Combined features shape: (180762, 40)
Combined labels shape: (180762,)


In [13]:
ros = RandomOverSampler(random_state=42, sampling_strategy='minority')
features_resampled, labels_resampled = ros.fit_resample(x_five_sec, y_five_sec)

print("Resampled features shape:", features_resampled.shape)
print("Resampled labels shape:", labels_resampled.shape)

Resampled features shape: (186377, 40)
Resampled labels shape: (186377,)


In [14]:
shuffle_idx = np.random.permutation(len(features_resampled))
X_combined = features_resampled[shuffle_idx]
y_combined = labels_resampled[shuffle_idx]

In [ ]:
def create_model(input_shape, num_classes):
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # Dense layers for classification
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

In [ ]:
def prepare_dataset(X, y, batch_size=32):
    """
    Convert numpy arrays to tf.data.Dataset with batching and prefetching
    """
    # Normalize features
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0) + 1e-8
    X_normalized = (X - X_mean) / X_std
    
    # Convert labels to categorical
    """label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    y_categorical = tf.keras.utils.to_categorical(y_encoded)"""
    
    # Create tf.data.Dataset
    #dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y_categorical))
    dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y))
    dataset = dataset.cache()  # Cache the data in memory
    dataset = dataset.shuffle(buffer_size=len(X))  # Shuffle the entire dataset
    dataset = dataset.batch(batch_size)  # Batch the data
    dataset = dataset.prefetch(tf.data.AUTOTUNE)  # Prefetch next batch
    
    return dataset, label_encoder

In [ ]:
def train_model_tpu(X, y, batch_size=32, epochs=50):
    """
    Train the model using TPU acceleration
    """
    # Initialize TPU
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        print('Running on TPU ', tpu.cluster_spec().as_dict()['worker'])
        
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        print("Number of accelerators: ", strategy.num_replicas_in_sync)
    except:
        print('No TPU detected. Using GPU/CPU strategy')
        strategy = tf.distribute.get_strategy()

    # Split the data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Prepare datasets
    train_dataset = prepare_dataset(X_train, y_train, batch_size)
    val_dataset = prepare_dataset(X_val, y_val, batch_size)
    
    # Get input shape and number of classes
    input_shape = (X.shape[1],)  # Will be (40,)
    num_classes = len(label_encoder.classes_)
    
    # Create and compile model using TPU strategy
    with strategy.scope():
        model = create_model(input_shape, num_classes)
        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=1e-6
        )
    ]
    
    # Train the model
    history = model.fit(
        train_dataset,
        epochs=epochs,
        validation_data=val_dataset,
        callbacks=callbacks
    )
    
    return model, history, label_encoder